<a href="https://colab.research.google.com/github/carolinampessoa/TechChallengeFase5/blob/main/TechChallengeFase5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1) Importação de bibliotecas**

Este bloco realiza a instalação das duas bibliotecas necessárias para execução do projeto no ambiente Google Colab:

*   OpenAI SDK: utilizada para acessar os modelos de linguagem (LLMs) responsáveis pela análise semântica da arquitetura apresentada na imagem.

*   ReportLab: biblioteca utilizada para geração automática de relatórios em formato PDF.

A instalação via pip garante que o ambiente de execução possua todas as dependências necessárias para executar as etapas de análise, enriquecimento e geração do relatório.

In [6]:
#Instalação de libs

!pip install openai
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.2 MB/s eta 0:00:00


**2) Configuração da API da LLM**

Configuração da autenticação necessária para utilizar os modelos de linguagem da OpenAI.

O acesso às APIs da OpenAI exige uma chave de autenticação (API Key).
O usuário informa sua chave manualmente no momento da execução, a qual é armazenada em uma variável de ambiente, chamada OPENAI_API_KEY. Isso permite que o cliente da OpenAI autentique automaticamente todas as chamadas feitas posteriormente no notebook.

Por questão de segurança, usamos a função getpass(), que evita que a chave fique visível no notebook ou registrada no histórico de execução.


In [7]:
#Configurar API Key (uso de LLM da OpenAI)

import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Digite sua OpenAI API Key: ")

Digite sua OpenAI API Key: ··········


**3) Upload da imagem de arquitetura**

Enviando a imagem a ser analisada.

Como o notebook é executado no Google Colab, não há acesso direto ao computador local do usuário. Portanto:

* A função files.upload() abre uma interface de upload;

* O usuário seleciona a imagem contendo o diagrama arquitetural;

* O arquivo é armazenado temporariamente no ambiente de execução;

O caminho do arquivo é então armazenado na variável image_path, que será utilizada nas etapas seguintes de processamento da imagem.

In [8]:
#Importar imagem para avaliação

from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Imagem carregada:", image_path)


Saving arch_02.png to arch_02.png
Imagem carregada: arch_02.png


**4) Funções auxiliares e preparação da imagem**

Preparando os recursos necessários para enviar a imagem ao modelo de linguagem:

* Inicialização do cliente da OpenAI;
* Conversão da imagem: a imagem enviada pelo usuário é convertida para Base64 para que possa ser analisada pelo modelo multimodal da LLM.

In [9]:
# Funções auxiliares

from openai import OpenAI
import base64
import json

client = OpenAI()

def encode_image(path):
    with open(path, "rb") as img:
        return base64.b64encode(img.read()).decode("utf-8")

base64_image = encode_image(image_path)

**5) Extração livre de componentes da arquitetura**

Identificando automaticamente os elementos arquiteturais presentes no diagrama.
É criado um prompt de análise arquitetural, solicitando ao modelo que examine o diagrama e identifique todos os elementos relevantes.

A instrução enviada ao modelo solicita que ele:

* Analise semanticamente o diagrama;

* Identifique componentes arquiteturais;

* Retorne os resultados em formato JSON estruturado

Essa etapa corresponde à extração automática de conhecimento arquitetural a partir da imagem.

In [10]:
# Extração Livre de Componentes

free_extraction_prompt = """
Analise o diagrama de arquitetura de software presente na imagem.

Identifique TODOS os elementos arquiteturais semanticamente relevantes.

Não restrinja a categorias pré-definidas.

Responda exclusivamente em JSON válido:

{
  "components": [
    {"name": "...", "type": "..."}
  ]
}
"""

response1 = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": free_extraction_prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

raw_components_text = response1.output_text
print(raw_components_text)


```json
{
  "components": [
    {
      "name": "User",
      "type": "Actor"
    },
    {
      "name": "Internet/Client",
      "type": "User Interface / Client Access"
    },
    {
      "name": "Microsoft Entra",
      "type": "Authentication Service"
    },
    {
      "name": "API Gateway",
      "type": "API Management Component"
    },
    {
      "name": "Developer Portal",
      "type": "API Management Interface"
    },
    {
      "name": "Logic Apps",
      "type": "Workflow and Orchestration Service"
    },
    {
      "name": "Azure Services",
      "type": "Backend System - Cloud Services"
    },
    {
      "name": "Saas Services",
      "type": "Backend System - Cloud Services"
    },
    {
      "name": "Web Services - REST",
      "type": "Backend System - Web API"
    },
    {
      "name": "Web Services - SOAP",
      "type": "Backend System - Web API"
    },
    {
      "name": "Resource Group",
      "type": "Azure Resource Management"
    }
  ]
}
```


**6) Limpeza e parsing da resposta**

Conversão da resposta da LLM em uma estrutura de dados manipulável pelo Python.
Modelos de linguagem frequentemente retornam JSON envolto em blocos markdown. Este bloco remove esses marcadores e realiza o parsing da resposta.

O processo ocorre em três etapas:

* Remoção de formatação Markdown.
* Conversão do texto JSON para objeto Python.
* Armazenamento na variável `raw_components`.

O resultado final é uma lista estruturada de componentes extraídos do diagrama.

In [11]:
# Limpeza + Parsing

clean_text = raw_components_text.strip()

if clean_text.startswith("```"):
    clean_text = clean_text.replace("```json", "").replace("```", "").strip()

raw_components = json.loads(clean_text)["components"]

raw_components


[{'name': 'User', 'type': 'Actor'},
 {'name': 'Internet/Client', 'type': 'User Interface / Client Access'},
 {'name': 'Microsoft Entra', 'type': 'Authentication Service'},
 {'name': 'API Gateway', 'type': 'API Management Component'},
 {'name': 'Developer Portal', 'type': 'API Management Interface'},
 {'name': 'Logic Apps', 'type': 'Workflow and Orchestration Service'},
 {'name': 'Azure Services', 'type': 'Backend System - Cloud Services'},
 {'name': 'Saas Services', 'type': 'Backend System - Cloud Services'},
 {'name': 'Web Services - REST', 'type': 'Backend System - Web API'},
 {'name': 'Web Services - SOAP', 'type': 'Backend System - Web API'},
 {'name': 'Resource Group', 'type': 'Azure Resource Management'}]

**7) Normalização taxonômica dos componentes**

Padronização dos tipos de componentes para uma taxonomia compatível com análise de segurança.
Os componentes identificados pela LLM podem possuir diversos nomes diferentes (EC2 Instance, Backend Server, API Gateway, etc). Para permitir a aplicação consistente do modelo STRIDE, os componentes são normalizados para as seguintes categorias padronizadas:

* user
* server
* database
* api
* external_system

Ou seja, essa etapa transforma descrições semânticas livres em **classes arquiteturais padronizadas**.


In [12]:
# Normalização Taxonômica

normalization_prompt = f"""
Normalize os componentes abaixo em categorias de segurança.

Categorias permitidas (escolha apenas UMA por item):

- user
- server
- database
- api
- external_system

Componentes:

{json.dumps(raw_components, indent=2)}

Responda em JSON válido:

{{
  "components": [
    {{"name": "...", "type": "..."}}
  ]
}}
"""

response2 = client.responses.create(
    model="gpt-4.1-mini",
    input=normalization_prompt
)

normalized_text = response2.output_text
print(normalized_text)


```json
{
  "components": [
    {"name": "User", "type": "user"},
    {"name": "Internet/Client", "type": "user"},
    {"name": "Microsoft Entra", "type": "external_system"},
    {"name": "API Gateway", "type": "api"},
    {"name": "Developer Portal", "type": "api"},
    {"name": "Logic Apps", "type": "server"},
    {"name": "Azure Services", "type": "server"},
    {"name": "Saas Services", "type": "external_system"},
    {"name": "Web Services - REST", "type": "api"},
    {"name": "Web Services - SOAP", "type": "api"},
    {"name": "Resource Group", "type": "server"}
  ]
}
```


**8) Parsing dos componentes normalizados**

Convertendo o resultado da normalização para estrutura de dados Python.
Assim como no bloco de parsing anterior, este trecho:

* Remove possíveis formatações de markdown.
* Converte o JSON retornado pela LLM.
* Armazena os resultados na variável `components`.

Cada componente passa a possuir uma estrutura como:

  {
  "name": "Amazon RDS",
  "type": "database"
  }

Essa estrutura será utilizada na etapa de análise de ameaças.


In [13]:
# Parsing Normalizado

clean_text = normalized_text.strip()

if clean_text.startswith("```"):
    clean_text = clean_text.replace("```json", "").replace("```", "").strip()

components = json.loads(clean_text)["components"]

components


[{'name': 'User', 'type': 'user'},
 {'name': 'Internet/Client', 'type': 'user'},
 {'name': 'Microsoft Entra', 'type': 'external_system'},
 {'name': 'API Gateway', 'type': 'api'},
 {'name': 'Developer Portal', 'type': 'api'},
 {'name': 'Logic Apps', 'type': 'server'},
 {'name': 'Azure Services', 'type': 'server'},
 {'name': 'Saas Services', 'type': 'external_system'},
 {'name': 'Web Services - REST', 'type': 'api'},
 {'name': 'Web Services - SOAP', 'type': 'api'},
 {'name': 'Resource Group', 'type': 'server'}]

**9) Aplicação do modelo STRIDE**

Associando possíveis ameaças de segurança aos componentes identificados, seguindo o modelo STRIDE.
Ou seja, este bloco define um mapa de ameaças STRIDE, relacionando os tipos de componentes identificados e normalizados às ameaças mais comuns.

Exemplo:

* database → Tampering, Information Disclosure
* server → Spoofing, Tampering, DoS

A função `analyze_stride()` percorre todos os componentes e associa as ameaças correspondentes.

O resultado produzido possui o formato:

{
component: "Amazon RDS",
type: "database",
threats: ["Tampering", "Information Disclosure"]
}

In [14]:
stride_map = {
    "server": ["Spoofing", "Tampering", "Denial of Service"],
    "database": ["Tampering", "Information Disclosure"],
    "api": ["Spoofing", "Repudiation"],
    "user": ["Spoofing"],
    "external_system": ["Spoofing", "Tampering"]
}

def analyze_stride(components):
    results = []

    for comp in components:
        threats = stride_map.get(comp["type"])
        if threats is None:
            threats = ["Unknown – No STRIDE mapping defined"]

        results.append({
            "component": comp["name"],
            "type": comp["type"],
            "threats": threats
        })

    return results

stride_results = analyze_stride(components)

stride_results


[{'component': 'User', 'type': 'user', 'threats': ['Spoofing']},
 {'component': 'Internet/Client', 'type': 'user', 'threats': ['Spoofing']},
 {'component': 'Microsoft Entra',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'API Gateway',
  'type': 'api',
  'threats': ['Spoofing', 'Repudiation']},
 {'component': 'Developer Portal',
  'type': 'api',
  'threats': ['Spoofing', 'Repudiation']},
 {'component': 'Logic Apps',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Azure Services',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Saas Services',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Web Services - REST',
  'type': 'api',
  'threats': ['Spoofing', 'Repudiation']},
 {'component': 'Web Services - SOAP',
  'type': 'api',
  'threats': ['Spoofing', 'Repudiation']},
 {'component': 'Resource Group',
  'type': 'se

**10) Enriquecimento das ameaças via LLM**

Adicionando explicações técnicas e contramedidas para cada ameaça identificada.
Os resultados da análise STRIDE são enviados novamente ao modelo de linguagem com um novo prompt, onde a LLM é instruída a:

* Explicar o racional técnico da ameaça
* Identificar os componentes afetados
* Sugerir contramedidas de segurança

Isso transforma uma simples lista de ameaças em uma análise de segurança detalhada. O texto gerado será utilizado diretamente no relatório final.


In [21]:
enrichment_prompt = f"""
Considere os resultados de STRIDE abaixo:

{json.dumps(stride_results, indent=2)}

Para cada ameaça, explique o racional técnico e sugira contramedidas.
Apresente a análise final terminando obrigatoriamente na seção 'Contramedidas', sem incluir comentários, frases de oferecimento de ajuda ou interações conversacionais após essa seção.

Resposta em texto estruturado.
"""

response3 = client.responses.create(
    model="gpt-4.1-mini",
    input=enrichment_prompt
)


text_output = response3.output_text
# Remove a última linha da resposta
lines = text_output.strip().split("\n")
text_output = "\n".join(lines[:-1])

print(text_output)


# Análise de ameaças STRIDE e contramedidas

---

## Spoofing

### Racional técnico
Spoofing refere-se à falsificação de identidade, onde um atacante finge ser uma entidade legítima para obter acesso não autorizado ou manipular o sistema. Componentes que expõem interfaces de autenticação, comunicação externa ou usuários são suscetíveis, podendo ser vítimas de ataques como falsificação de identidade, credenciais roubadas ou uso de tokens falsificados.

### Componentes afetados
- User
- Internet/Client
- Microsoft Entra
- API Gateway
- Developer Portal
- Logic Apps
- Azure Services
- Saas Services
- Web Services - REST
- Web Services - SOAP
- Resource Group

### Contramedidas
- Implementar autenticação forte (multi-fator MFA) para usuários e sistemas.
- Utilizar protocolos de autenticação e autorização robustos (OAuth 2.0, OpenID Connect).
- Utilizar certificados digitais e criptografia para validar identidade de serviços.
- Aplicar validação rigorosa de tokens e sessões com expiração e 

**11) Geração do relatório em PDF**

Geração automática do relatório técnico com os resultados da análise. Este bloco utiliza a biblioteca **ReportLab** para criar um documento PDF contendo:

- Título do relatório
- Seções de ameaças
- Explicações técnicas
- Contramedidas sugeridas

Também são definidos estilos visuais para melhorar a legibilidade, incluindo:

- Títulos
- Subtítulos
- Parágrafos
- Listas

O conteúdo gerado pela LLM é formatado e inserido no documento, que em seguida é salvo diretamente no Google Drive do usuário.

In [22]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

pdf_path = "relatorio_stride.pdf"

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=50,
    leftMargin=50,
    topMargin=50,
    bottomMargin=50
)

styles = getSampleStyleSheet()

# ===== ESTILOS =====

title_style = ParagraphStyle(
    'Title',
    parent=styles['Title'],
    fontSize=22,
    textColor=colors.darkblue,
    spaceAfter=20
)

section_style = ParagraphStyle(
    'Section',
    parent=styles['Heading2'],
    fontSize=16,
    textColor=colors.darkblue,
    spaceAfter=12
)

subtitle_style = ParagraphStyle(
    'Subtitle',
    parent=styles['Heading3'],
    fontSize=12,
    textColor=colors.black,
    spaceAfter=6
)

body_style = ParagraphStyle(
    'Body',
    parent=styles['BodyText'],
    fontSize=11.5,
    leading=16,
    spaceAfter=8
)

bullet_style = ParagraphStyle(
    'Bullet',
    parent=body_style,
    leftIndent=20,
    spaceAfter=4
)

elements = []

text_output = response3.output_text

elements.append(Paragraph("Relatório de Ameaças – STRIDE", title_style))

for line in text_output.split("\n"):

    line = line.strip()

    if not line:
        elements.append(Spacer(1, 6))
        continue

    # Separadores visuais
    if line.startswith("---"):
        elements.append(Spacer(1, 12))
        continue

    # Títulos principais (###)
    if line.startswith("###"):
        clean = line.replace("###", "").strip()
        elements.append(Spacer(1, 10))
        elements.append(Paragraph(f"<b>{clean}</b>", section_style))
        continue

    # Subtítulos em negrito
    if line.startswith("**") and line.endswith("**"):
        clean = line.replace("**", "")
        elements.append(Paragraph(f"<b>{clean}</b>", subtitle_style))
        continue

    if line.startswith("**"):
        clean = line.replace("**", "").replace(":", "")
        elements.append(Paragraph(f"<b>{clean}:</b>", subtitle_style))
        continue

    # Bullet points
    if line.startswith("-"):
        clean = line[1:].strip()
        elements.append(Paragraph(f"• {clean}", bullet_style))
        continue

    # Texto normal
    elements.append(Paragraph(line, body_style))

doc.build(elements)


In [23]:
pdf_path = "/content/drive/MyDrive/relatorio_stride.pdf"

**12) Validação de Resultados**

Para validar a abordagem proposta, foram utilizados cinco diagramas de arquitetura distintos contendo diferentes combinações de elementos e serviços. Para cada imagem foi definido previamente um conjunto de componentes esperados, utilizado como referência (ground truth).

O modelo foi então aplicado para identificar automaticamente os componentes presentes nos diagramas. Os resultados obtidos foram comparados com os componentes reais utilizando métricas clássicas de avaliação em tarefas de identificação:

* Precisão (Precision): proporção de componentes identificados corretamente entre todos os componentes detectados pelo modelo.

* Revocação (Recall): proporção de componentes corretos identificados em relação ao total de componentes existentes.

* F1-score: média harmônica entre precisão e recall, utilizada como indicador geral de desempenho.

Essa validação permite avaliar quantitativamente a capacidade do modelo de identificar corretamente os elementos arquiteturais presentes nos diagramas analisados.

In [26]:
validation_dataset = [
    {
        "image": "arch_01.png",
        "expected_components": [
            "AWS Shield",
            "Amazon CloudFront",
            "AWS WAF",
            "Application Load Balancer",
            "Amazon RDS"
        ]
    },
    {
        "image": "arch_02.png",
        "expected_components": [
            "Amazon S3",
            "AWS Lambda",
            "Amazon API Gateway",
            "Amazon CloudWatch"
        ]
    },
    {
        "image": "arch_03.png",
        "expected_components": [
            "Amazon EC2",
            "Application Load Balancer",
            "Amazon RDS",
            "Amazon ElastiCache"
        ]
    },
    {
        "image": "arch_04.png",
        "expected_components": [
            "Amazon SES",
            "Amazon CloudWatch",
            "AWS CloudTrail"
        ]
    },
    {
        "image": "arch_05.png",
        "expected_components": [
            "AWS Shield",
            "AWS WAF",
            "CloudFront",
            "VPC"
        ]
    }
]

In [27]:
def evaluate_components(expected, predicted):

    expected_set = set([e.lower() for e in expected])
    predicted_set = set([p.lower() for p in predicted])

    true_positive = expected_set.intersection(predicted_set)
    false_positive = predicted_set - expected_set
    false_negative = expected_set - predicted_set

    precision = len(true_positive) / len(predicted_set) if predicted_set else 0
    recall = len(true_positive) / len(expected_set) if expected_set else 0

    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_positive": list(true_positive),
        "false_positive": list(false_positive),
        "false_negative": list(false_negative)
    }

In [31]:
import json

def run_component_detection(image_path):
    # Ensure 'encode_image', 'client', and 'free_extraction_prompt' are accessible from global scope

    # Encode the image to base64
    base64_image = encode_image(image_path)

    # Call the OpenAI API to extract components
    response1 = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": free_extraction_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{base64_image}"
                    }
                ]
            }
        ]
    )

    raw_components_text = response1.output_text

    # Clean and parse the JSON response
    clean_text = raw_components_text.strip()
    if clean_text.startswith("```"):
        clean_text = clean_text.replace("```json", "").replace("```", "").strip()

    raw_component_dicts = json.loads(clean_text)["components"]

    # Extract only the component names for evaluation purposes
    predicted_component_names = [comp["name"] for comp in raw_component_dicts]
    return predicted_component_names

results = []

for sample in validation_dataset:

    image_path = sample["image"]
    expected = sample["expected_components"]

    predicted = run_component_detection(image_path)

    metrics = evaluate_components(expected, predicted)

    results.append({
        "image": image_path,
        "expected": expected,
        "predicted": predicted,
        "metrics": metrics
    })


In [32]:
for r in results:

    print("Imagem:", r["image"])
    print("Componentes esperados:", r["expected"])
    print("Componentes detectados:", r["predicted"])

    print("Precision:", round(r["metrics"]["precision"], 2))
    print("Recall:", round(r["metrics"]["recall"], 2))
    print("F1 Score:", round(r["metrics"]["f1_score"], 2))

    print("-" * 40)

Imagem: arch_01.png
Componentes esperados: ['AWS Shield', 'Amazon CloudFront', 'AWS WAF', 'Application Load Balancer', 'Amazon RDS']
Componentes detectados: ['Usuários SEI', 'AWS Shield', 'Amazon CloudFront', 'AWS WAF', 'AWS Cloud', 'sa-east-1 (São Paulo)', 'Virtual Private Cloud', 'Availability Zone A', 'Availability Zone B', 'Availability Zone C', 'Public Subnet', 'Private Subnet', 'Application Load Balancer', 'SEI / SIP', 'Auto Scaling (API Server)', 'Auto Scaling (Solr)', 'Solr', 'Amazon Elastic File System (NFS) Multi-AZ', 'Amazon RDS (Primary)', 'Amazon RDS (Secondary)', 'Amazon ElastiCache (memcached) Multi-AZ', 'AWS CloudTrail', 'AWS Key Management Service', 'AWS Backup', 'Amazon CloudWatch', 'Amazon Simple Email Service (SES)']
Precision: 0.15
Recall: 0.8
F1 Score: 0.26
----------------------------------------
Imagem: arch_02.png
Componentes esperados: ['Amazon S3', 'AWS Lambda', 'Amazon API Gateway', 'Amazon CloudWatch']
Componentes detectados: ['User', 'Microsoft Entra', 'AP

In [33]:
import pandas as pd

rows = []

for r in results:
    rows.append({
        "Imagem": r["image"],
        "Precision": r["metrics"]["precision"],
        "Recall": r["metrics"]["recall"],
        "F1 Score": r["metrics"]["f1_score"]
    })

df_results = pd.DataFrame(rows)

df_results

,Imagem,Precision,Recall,F1 Score
0,arch_01.png,0.153846,0.80,0.258065
1,arch_02.png,0.000000,0.00,0.000000
2,arch_03.png,0.000000,0.00,0.000000
3,arch_04.png,0.000000,0.00,0.000000
4,arch_05.png,0.050000,0.25,0.083333


**13) Considerações Finais**

O presente trabalho demonstrou a aplicação de modelos de linguagem de grande escala (LLMs) para auxiliar na identificação automatizada de ameaças de segurança em arquiteturas de sistemas computacionais. A solução proposta utiliza um fluxo que integra análise visual de diagramas arquiteturais, extração semântica de componentes, normalização taxonômica e aplicação do modelo de ameaças STRIDE para geração automática de um relatório técnico contendo explicações e contramedidas de segurança.

O pipeline desenvolvido evidencia o potencial da inteligência artificial generativa como ferramenta de apoio em processos de Threat Modeling, atividade tradicionalmente conduzida de forma manual por especialistas em segurança. Ao automatizar etapas como a identificação de componentes arquiteturais através de imagens e a associação de ameaças relevantes, a solução contribui para reduzir o esforço humano necessário e aumentar a velocidade de análise de sistemas complexos.

Entretanto, sendo um MVP, o fluxo proposto também apresenta algumas limitações que devem ser consideradas. Primeiramente, a precisão da identificação dos componentes depende diretamente da qualidade do diagrama fornecido e da capacidade do modelo de linguagem em interpretar corretamente os elementos visuais presentes na imagem. Diagramas excessivamente complexos, com baixa resolução ou contendo simbologias não padronizadas, podem resultar em interpretações incorretas ou incompletas.

Também deve ser considerado que o mapeamento entre componentes arquiteturais e ameaças foi realizado com base em categorias simplificadas. Em ambientes reais, arquiteturas podem apresentar interações mais complexas, envolvendo múltiplas camadas de infraestrutura, serviços gerenciados e integrações externas que exigiriam uma modelagem de ameaças mais detalhada.

Como possíveis melhorias futuras, diversas extensões podem ser exploradas. Uma evolução natural seria a integração com modelos especializados em computer vision ou detecção de objetos para melhorar a identificação estrutural dos elementos presentes nos diagramas. Outra possibilidade seria a incorporação de ontologias de arquitetura de software e segurança, permitindo uma classificação mais precisa dos componentes e relações entre eles.
